TASK 4 – OPTIMAL FICO BINNING USING LOG-LIKELIHOOD
Master-Level Credit Risk Segmentation

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. LOAD DATA

# Update path if necessary
file_path = "Task 3 and 4_Loan_Data.csv"
df = pd.read_csv(file_path)

# Identify FICO and default columns automatically
fico_col = [col for col in df.columns if "fico" in col.lower()][0]
default_col = [col for col in df.columns if "default" in col.lower()][0]

# Keep only relevant columns
df = df[[fico_col, default_col]].dropna()

# Sort by FICO score (required for ordered segmentation)
df = df.sort_values(by=fico_col).reset_index(drop=True)

print(f"Dataset size: {len(df)}")


Dataset size: 10000


In [3]:
# 2. AGGREGATE BY UNIQUE FICO SCORE
# Reduces computational complexity significantly

agg = df.groupby(fico_col)[default_col].agg(
    n_obs="count",
    n_defaults="sum"
).reset_index()

agg = agg.sort_values(by=fico_col).reset_index(drop=True)

print(f"Unique FICO levels: {len(agg)}")

Unique FICO levels: 374


In [4]:
# 3. DEFINE BINOMIAL LOG-LIKELIHOOD FUNCTION

def compute_log_likelihood(n, k):
    """
    Compute the binomial log-likelihood contribution for one bin.

    Parameters:
        n (int): total observations in bin
        k (int): number of defaults in bin

    Returns:
        float: log-likelihood value
    """
    # Numerical stability adjustment
    if k == 0 or k == n:
        p = (k + 1e-8) / (n + 2e-8)
    else:
        p = k / n

    return k * np.log(p) + (n - k) * np.log(1 - p)


In [5]:
# 4. OPTIMAL BINNING VIA DYNAMIC PROGRAMMING

def optimal_binning(agg_data, n_bins):
    """
    Perform supervised optimal binning using dynamic programming
    to maximize total log-likelihood.

    Parameters:
        agg_data (DataFrame): aggregated FICO-level data
        n_bins (int): desired number of rating categories

    Returns:
        list of tuples: bin index ranges
    """
    m = len(agg_data)

    # Initialize DP tables
    dp = np.full((m + 1, n_bins + 1), -np.inf)
    split = np.zeros((m + 1, n_bins + 1), dtype=int)

    dp[0][0] = 0  # Base condition

    # Fill DP table
    for i in range(1, m + 1):
        for j in range(1, n_bins + 1):
            for k in range(j - 1, i):

                # Compute interval statistics
                segment = agg_data.iloc[k:i]
                n_obs = segment["n_obs"].sum()
                n_defaults = segment["n_defaults"].sum()

                ll = compute_log_likelihood(n_obs, n_defaults)

                if dp[k][j - 1] + ll > dp[i][j]:
                    dp[i][j] = dp[k][j - 1] + ll
                    split[i][j] = k

    # Backtrack to recover optimal bins
    bins = []
    i = m
    j = n_bins

    while j > 0:
        k = split[i][j]
        bins.append((k, i))
        i = k
        j -= 1

    bins.reverse()
    return bins


In [6]:
# 5. APPLY OPTIMAL BINNING

n_bins = 5  # Number of rating categories
bins = optimal_binning(agg, n_bins)

In [7]:
# 6. BUILD FINAL RATING TABLE

# Define rating scale (A = best credit quality)
rating_scale = ["A", "B", "C", "D", "E"]

results = []

for idx, (start, end) in enumerate(bins):

    segment = agg.iloc[start:end]

    min_fico = segment[fico_col].min()
    max_fico = segment[fico_col].max()

    total_obs = segment["n_obs"].sum()
    total_defaults = segment["n_defaults"].sum()

    pd_rate = total_defaults / total_obs

    results.append({
        "FICO Range": f"{min_fico} - {max_fico}",
        "Rating": rating_scale[::-1][idx],  # Reverse so best FICO gets A
        "Observations": int(total_obs),
        "Defaults": int(total_defaults),
        "Probability of Default (PD)": round(pd_rate, 4)
    })

rating_table = pd.DataFrame(results)

print("\n==========================================")
print("OPTIMAL FICO RATING TABLE")
print("==========================================\n")
print(rating_table)


OPTIMAL FICO RATING TABLE

  FICO Range Rating  Observations  Defaults  Probability of Default (PD)
0  408 - 520      E           301       199                       0.6611
1  521 - 580      D          1407       536                       0.3810
2  581 - 640      C          3438       703                       0.2045
3  641 - 696      B          3197       336                       0.1051
4  697 - 850      A          1657        77                       0.0465
